In [ ]:
EXPECTED_FILES={'adapter_config.json': {'bytes': 1227, 'sha256': 'a6aec727ab26ba005e55e1c84145032a7ad8cdb8a3e21a3d3d8ee9c0e3b7792f'}, 'adapter_model.safetensors': {'bytes': 466767600, 'sha256': 'd43af1abea2c99c6fe527546c72bcb6acdb4cdb972a6dc9181a7b42e007e7b30'}, 'export-report.json': {'bytes': 10870, 'sha256': '4bb5dec1e460c504d66695b56d37617c84f6c42882ce341426a5b1e3efe425a1'}}
EXPECTED_REPORT_SHA256='7c2129b8b3d81a2ccc633a28129e6d2300f8c03c7668f11be1aeafa3d770e50d'
from pathlib import Path
from hashlib import sha256
import json,struct,time
roots=[Path('/kaggle/input/notebooks/kunalsinghh25/arc-hydra-qwen27-lora-pilot/1'),
    Path('/kaggle/input/notebooks/kunalsinghh25/arc-hydra-qwen27-lora-pilot'),
    Path('/kaggle/input/arc-hydra-qwen27-lora-pilot')]
hits=set()
for root in roots:
    if root.is_dir():hits.update(p.parent.resolve() for p in root.rglob('serving-adapter/adapter_config.json'))
if len(hits)!=1:raise ValueError('Pinned training adapter not uniquely mounted: '+str(hits))
adapter=next(iter(hits))
def digest(path):
    value=sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda:stream.read(8*1024*1024),b''):value.update(chunk)
    return value.hexdigest()
for name,record in EXPECTED_FILES.items():
    path=adapter/name
    assert path.stat().st_size==record['bytes'] and digest(path)==record['sha256'],name
assert digest(adapter.parent/'training-report.json')==EXPECTED_REPORT_SHA256
config=json.loads((adapter/'adapter_config.json').read_text())
assert config['r']==16 and config['lora_alpha']==32
assert sorted(config['target_modules'])==sorted(['qkv_proj','o_proj','out_proj','in_proj_qkvz','gate_up_proj','down_proj'])
with (adapter/'adapter_model.safetensors').open('rb') as stream:
    length=struct.unpack('<Q',stream.read(8))[0]
    assert 0<length<2**20
    header=json.loads(stream.read(length))
tensors={name:value for name,value in header.items() if name!='__metadata__'}
assert len(tensors)==992 and all(value['dtype']=='F32' for value in tensors.values())
qkvz=[]
for name,value in tensors.items():
    if '.in_proj_q.' not in name or '.lora_B.' not in name:continue
    for projection,width in [('q',2048),('k',2048),('v',6144),('z',6144)]:
        b=name.replace('.in_proj_q.','.in_proj_'+projection+'.')
        a=b.replace('.lora_B.','.lora_A.')
        assert tensors[b]['shape']==[width,16] and tensors[a]['shape']==[16,5120],b
    qkvz.append(name)
assert len(qkvz)==48
result=dict(passed=True,training_kernel='kunalsinghh25/arc-hydra-qwen27-lora-pilot/1',
    adapter_directory=str(adapter),actual_tensor_count=len(tensors),recurrent_groups_checked=len(qkvz),
    training_report_sha256=EXPECTED_REPORT_SHA256,adapter_files=EXPECTED_FILES,
    boundary='CPU-only exact mount, file hashes and saved tensor-header check. No weights loaded into a model; no inference or gameplay claim.')
Path('/kaggle/working/adapter-mount-probe.json').write_text(json.dumps(result,indent=2)+'\n')
print(json.dumps(result,indent=2))
